# Memory Recall Evaluation with TruLens

This notebook demonstrates how to evaluate an AI agent's memory recall using
TruLens' existing `recall_at_k`, `precision_at_k`, and `mrr` metrics with
`conversation_id` scoping.

## The Problem

Agents with memory stores need to surface the **right** memories at the right
time. When a user asks a follow-up question, the agent must:
1. **Recall** relevant memories (high recall)
2. **Avoid** retrieving irrelevant ones (high precision)
3. **Rank** the most relevant memories first (high MRR)

## The Approach

We use `GroundTruthAgreement` with `conversation_id` to scope ground truth
lookups per conversation. Memory texts go into `expected_chunks` — the same
format used for RAG chunk evaluation — so there's one unified ground truth
format.

## The Story

We'll walk through a full debugging loop:
1. Set up a golden set and a "broken" agent that retrieves poorly
2. Score it — low recall/precision/MRR
3. Fix the retrieval strategy
4. Re-score — scores improve

In [ ]:
from trulens.feedback.groundtruth import GroundTruthAgreement
from unittest.mock import MagicMock

## 1. Define the Golden Set

Each entry has a `query`, `expected_chunks` (memory texts), and an optional
`conversation_id` to scope lookups.

In [ ]:
golden_set = [
    {
        "query": "What did the user say about the project deadline?",
        "expected_chunks": [
            {"text": "User said the deadline is end of March", "expect_score": 1},
            {"text": "User wants to review the design first", "expect_score": 1},
        ],
        "conversation_id": "conv_1",
    },
    {
        "query": "What are the user's preferences?",
        "expected_chunks": [
            {"text": "User prefers dark mode", "expect_score": 1},
            {"text": "User likes Python", "expect_score": 1},
        ],
        "conversation_id": "conv_1",
    },
    # Same query in a different conversation — different expected memories
    {
        "query": "What are the user's preferences?",
        "expected_chunks": [
            {"text": "User prefers light mode", "expect_score": 1},
        ],
        "conversation_id": "conv_2",
    },
]

In [ ]:
# Create a GTA with a mock provider (replace with real provider in production)
provider = MagicMock()
provider.engine_model = "test-model"
gta = GroundTruthAgreement(golden_set, provider=provider)

## 2. Before: The Broken Agent

Our agent has a memory store but its retrieval is poor — it returns irrelevant
memories, misses important ones, and ranks wrong results first.

Let's evaluate it against both queries in `conv_1`.

In [ ]:
def evaluate(gta, query, retrieved, conversation_id):
    """Run all three metrics and print results."""
    recall = gta.recall_at_k(query, retrieved, conversation_id=conversation_id)
    precision = gta.precision_at_k(query, retrieved, conversation_id=conversation_id)
    mrr_val = gta.mrr(query, retrieved, conversation_id=conversation_id)
    print(f"  recall: {recall:.2f}  precision: {precision:.2f}  mrr: {mrr_val:.2f}")
    return recall, precision, mrr_val

In [ ]:
print("=== BEFORE: Broken agent retrieval ===")
print()

# Query 1: Agent returns irrelevant meeting notes instead of deadline info
print("Q: What did the user say about the project deadline?")
print("  Retrieved: [Meeting notes from Tuesday, Budget is $50k]")
before_q1 = evaluate(
    gta,
    "What did the user say about the project deadline?",
    ["Meeting notes from Tuesday", "Budget is $50k"],
    conversation_id="conv_1",
)
print()

# Query 2: Agent finds one relevant memory but buries it under irrelevant ones
print("Q: What are the user's preferences?")
print("  Retrieved: [Schedule standup for 9am, User prefers dark mode, Deploy v2.1]")
before_q2 = evaluate(
    gta,
    "What are the user's preferences?",
    ["Schedule standup for 9am", "User prefers dark mode", "Deploy v2.1"],
    conversation_id="conv_1",
)

**What went wrong?**
- **Query 1**: Recall is 0.00 — the agent completely missed both expected memories.
- **Query 2**: Recall is 0.50 (found 1 of 2), precision is 0.33 (1 of 3 relevant),
  MRR is 0.50 (the one relevant result is at rank 2).

The metrics tell us two different stories: the first query needs *more recall*,
the second needs *better ranking and fewer false positives*.

## 3. Diagnose and Fix

Based on the scores:

- **Query 1 failure** → The retrieval step isn't matching the query to the right
  memories at all. Fix: improve the embedding/query or lower the similarity
  threshold so relevant memories surface.

- **Query 2 failure** → Relevant memories exist in the store but are drowned out
  by noise. Fix: add a re-ranking step or increase `k` in `recall_at_k` to
  check if the right memories appear further down the list.

After adjusting the retrieval pipeline, we get better results:

In [ ]:
print("=== AFTER: Fixed agent retrieval ===")
print()

# Query 1: Now the agent retrieves the correct deadline memories
print("Q: What did the user say about the project deadline?")
print("  Retrieved: [User said the deadline is end of March, User wants to review the design first]")
after_q1 = evaluate(
    gta,
    "What did the user say about the project deadline?",
    ["User said the deadline is end of March", "User wants to review the design first"],
    conversation_id="conv_1",
)
print()

# Query 2: Now relevant memories are ranked first, no noise
print("Q: What are the user's preferences?")
print("  Retrieved: [User prefers dark mode, User likes Python]")
after_q2 = evaluate(
    gta,
    "What are the user's preferences?",
    ["User prefers dark mode", "User likes Python"],
    conversation_id="conv_1",
)

## 4. Compare Before vs After

| Query | Metric | Before | After |
|-------|--------|--------|-------|
| Deadline | recall | 0.00 | 1.00 |
| Deadline | precision | 0.00 | 1.00 |
| Deadline | mrr | 0.00 | 1.00 |
| Preferences | recall | 0.50 | 1.00 |
| Preferences | precision | 0.33 | 1.00 |
| Preferences | mrr | 0.50 | 1.00 |

The metrics made the failure visible and confirmed the fix worked. That's the
debugging loop: **measure → diagnose → fix → re-measure**.

## 5. Conversation Scoping

The same query can have different expected memories in different conversations.
`conversation_id` ensures each evaluation only considers the right ground truth.

Without scoping, the same query would silently union every conversation's
expected chunks — inflating the recall denominator and giving wrong scores.

In [ ]:
# Same query, conv_1 expects dark mode + Python
recall_conv1 = gta.recall_at_k(
    "What are the user's preferences?",
    ["User prefers dark mode", "User likes Python"],
    conversation_id="conv_1",
)

# Same query, conv_2 expects light mode — these are the WRONG memories for conv_2
recall_conv2 = gta.recall_at_k(
    "What are the user's preferences?",
    ["User prefers dark mode", "User likes Python"],
    conversation_id="conv_2",
)

print(f"Conv 1 recall: {recall_conv1:.2f}")  # 1.00
print(f"Conv 2 recall: {recall_conv2:.2f}")  # 0.00 — wrong memories for conv_2

## 6. Per-call vs Instance-level conversation_id

You can set `conversation_id` on the constructor as a default (useful when
evaluating a single conversation), and override it per-call when needed.

In [ ]:
# Instance-level default: always scope to conv_1
gta_conv1 = GroundTruthAgreement(golden_set, provider=provider, conversation_id="conv_1")

# Uses the instance default (conv_1)
r_default = gta_conv1.recall_at_k(
    "What are the user's preferences?",
    ["User prefers dark mode", "User likes Python"],
)

# Per-call override: evaluate against conv_2 instead
r_override = gta_conv1.recall_at_k(
    "What are the user's preferences?",
    ["User prefers light mode"],
    conversation_id="conv_2",
)

print(f"Default (conv_1): {r_default:.2f}")   # 1.00
print(f"Override (conv_2): {r_override:.2f}")  # 1.00